## case study 1 :  Hospital Readmission Prediction — Use logistic regression with L2 regularization on patient records (diagnosis codes, vitals, prior visits) to predict 30-day readmission risk. Evaluate with ROC-AUC and discuss clinical cost of false negatives vs. false positives.**italicised text**

In [2]:
from google.colab import files

uploaded = files.upload()

Saving hospital_readmissions_30k.csv to hospital_readmissions_30k.csv


In [8]:
# 1. Import libraries
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

# 2. Load dataset
data = pd.read_csv("hospital_readmissions_30k.csv")

# 3. Display first 5 rows
print(data.head())

# 4. Check dataset information
print(data.info())

# 5. Convert Yes/No columns into 1/0
data["diabetes"] = data["diabetes"].map({"Yes": 1, "No": 0})
data["hypertension"] = data["hypertension"].map({"Yes": 1, "No": 0})
data["readmitted_30_days"] = data["readmitted_30_days"].map({
    "Yes": 1,
    "No": 0
})

# 6. Convert Blood Pressure into numerical values
data[["systolic", "diastolic"]] = data["blood_pressure"].str.split(
    "/", expand=True
).astype(float)

# 7. Convert categorical columns into numbers
data = pd.get_dummies(
    data,
    columns=["gender", "discharge_destination"],
    drop_first=True
)

# 8. Select features
X = data.drop(
    columns=[
        "patient_id",
        "blood_pressure",
        "readmitted_30_days"
    ]
)

# 9. Target
y = data["readmitted_30_days"]

# 10. Split data into training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 11. Standardization
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 12. Logistic Regression with L2 Regularization
model = LogisticRegression(
    penalty="l2",
    C=1.0,
    max_iter=1000
)

# 13. Train model
model.fit(X_train, y_train)

# 14. Predict probability
y_probability = model.predict_proba(X_test)[:, 1]

# 15. ROC-AUC
roc_auc = roc_auc_score(y_test, y_probability)

print("\nROC-AUC Score:", roc_auc)

# 16. Prediction using 0.5 threshold
y_prediction = (y_probability >= 0.5).astype(int)

# 17. Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_prediction))

   patient_id  age  gender blood_pressure  cholesterol   bmi diabetes  \
0           1   74   Other         130/72          240  31.5      Yes   
1           2   46  Female         120/92          292  36.3       No   
2           3   89   Other         135/78          153  30.3       No   
3           4   84  Female         123/80          153  31.5       No   
4           5   32   Other         135/84          205  18.4       No   

  hypertension  medication_count  length_of_stay discharge_destination  \
0           No                 5               1      Nursing_Facility   
1           No                 4               3      Nursing_Facility   
2          Yes                 1               1                  Home   
3          Yes                 3              10                  Home   
4          Yes                 6               4      Nursing_Facility   

  readmitted_30_days  
0                Yes  
1                 No  
2                 No  
3                 No  
4

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
